# N02 Main Evaluation — v1.0.3 re-freeze (replaces the 7 code texts that v1.0.2 got wrong)

**Why v1.0.3:** in v1.0.2 the code variants of T039, T041, T048, T067, T085, T088, T090 were copied from *other* topics (turbine vibration, network delay, CPU metrics, ...), did not match the topic's `operation_id` (min-max normalization), and T067 was a truncated function. Also the manifest `role` column disagreed with the report.

**What this notebook does**
1. Loads the **v1.0.2 CSV** and rebuilds the 7 code texts from each topic's *own* phrase (taken from its `general_prose` text) and its original sample values (verified against the v1.0 candidate CSV), with min-max code.
2. Recomputes **all** checks from scratch on the edited data (duplicates, both tokenizers, syntax, truncated functions, topic phrase, min-max structure). Nothing is reused from earlier cells.
3. Writes a `candidate` manifest unless every hard check passes **and** a person sets the human-review flag. `role` in the CSV always agrees with `frozen_for_confirmatory_analysis` in the report.

**Human must confirm before freezing:** (a) the 7 replaced quartets read as the same topic in all four styles; (b) the warnings printed in step 4 (other code texts that carry little topic information) are acceptable or fixed. Medium replacements omit the equal-values guard to stay under the GPT-2 128-token cap — a small format difference from the long tier.

In [ ]:
%pip -q install -U transformers pandas

## 1. Upload the v1.0.2 manifest (`N02_Main_Evaluation_FROZEN_Manifest_v1.0.2.csv`)

In [ ]:
import hashlib
from pathlib import Path
import pandas as pd
p = Path('/content/N02_Main_Evaluation_FROZEN_Manifest_v1.0.2.csv')
if not p.exists():
    from google.colab import files
    up = files.upload()
    p = Path('/content') / [n for n in up if n.lower().endswith('.csv')][0]
SOURCE_SHA256 = hashlib.sha256(p.read_bytes()).hexdigest()
manifest_v102 = pd.read_csv(p)
print(p.name, '| rows', len(manifest_v102), '| sha256', SOURCE_SHA256)
# expected for the uploaded v1.0.2 file: a9f4a23c631857dcbe80c76e3ee6732ee956ce5b007b5e1ea542da7b09ca8f48

## 2. Tokenizers (both are needed: GPT-2 counts every indent space, Pythia does not)

In [ ]:
from transformers import AutoTokenizer
tokenizers = {'gpt2': AutoTokenizer.from_pretrained('gpt2'),
              'pythia-160m': AutoTokenizer.from_pretrained('EleutherAI/pythia-160m')}

## 3. Replacement texts and checks (definitions)

In [ ]:
# ---------- CELL A: replacement code texts (each rebuilt from its own topic's phrase / operation) ----------
import re, ast, json, hashlib
from pathlib import Path
import pandas as pd

# topic phrase = the exact object phrase used in that topic's general_prose text ("scale the <PHRASE> to the zero-to-one interval ...")
REPLACE_IDS = ['T039','T041','T048','T067','T085','T088','T090']
SAMPLES = {   # sample values = the originals from the v1.0 candidate manifest (verified against the candidate CSV)
    'T039': '[18.4, 19.1, 17.8, 20.0]', 'T041': '[2.1, 3.4, 2.9, 4.0]', 'T048': '[205, 198, 212, 220]',
    'T067': '[12, 15, 9, 17]',
    'T085': '[3.8, 3.6, 3.9, 3.7]',      'T088': '[20, 35, 18, 42]', 'T090': '[4.2, 4.0, 5.1, 4.8]',
}

def topic_phrase(manifest, topic_id):
    prose = manifest[(manifest.topic_id == topic_id) & (manifest['style'] == 'general_prose')]['text'].iloc[0]
    m = re.search(r'scale the (.*?) to the zero-to-one interval', prose)
    if not m: raise ValueError(f'{topic_id}: cannot find topic phrase in general_prose')
    return m.group(1)

def build_medium(tid, phrase):
    # 4-space indent like the other medium code texts; the equal-values guard is omitted to stay inside the GPT-2 128-token cap
    return (f"def process(values):\n    if not values:\n        return None\n"
            f"    low, high = min(values), max(values)\n"
            f"    return [(value-low)/(high-low) for value in values]\n\n"
            f"sample = {SAMPLES[tid]}\nresult = process(sample)\nprint(result)\n"
            f"# Min-max scale the {phrase}.")

def build_long(tid, phrase):
    return (f"def analyze(values):\n    if not values:\n        return None\n"
            f"    low, high = min(values), max(values)\n    if high == low:\n        return [0.0 for _ in values]\n"
            f"    return [(value - low) / (high - low) for value in values]\n\n"
            f"values = {SAMPLES[tid]}\nresult = analyze(values)\nprint(\"Input:\", values)\nprint(\"Result:\", result)\n"
            f"# Context: these observations represent {phrase}.\n"
            f"# Task: scale the {phrase} to the zero-to-one interval with min-max normalization.\n"
            f"# Interpretation: normalization expresses values on a common numerical scale for selected comparisons.\n"
            f"# Validation: the transformation depends on the observed minimum and maximum and needs an explicit policy when all values are equal.\n"
            f"# Explain whether the code and the interpretation match the intended measurement workflow.")

def build_replacements(manifest):
    out = {}
    for tid in REPLACE_IDS:
        tier = manifest[manifest.topic_id == tid]['length_tier'].iloc[0]
        ph = topic_phrase(manifest, tid)
        out[f'{tid}_code_plus_natural_language'] = (build_long if tier == 'long' else build_medium)(tid, ph)
    return out


In [ ]:
# ---------- CELL B: apply replacements, then recompute ALL checks fresh (nothing reused from earlier cells) ----------
LENGTH_BANDS = {'short': (16, 64), 'medium': (65, 128), 'long': (129, 256)}
TOKENIZER_IDS = {'gpt2': 'gpt2', 'pythia-160m': 'EleutherAI/pythia-160m'}
EXPECTED_STYLES = {'general_prose', 'question_dialogue', 'academic', 'code_plus_natural_language'}

def is_valid_python(t):
    try: ast.parse(t); return True
    except SyntaxError: return False

def function_is_complete(t):
    """every top-level def must end with a return/raise (catches truncated functions); plain scripts (no def) pass"""
    try: tree = ast.parse(t)
    except SyntaxError: return False
    fns = [n for n in tree.body if isinstance(n, ast.FunctionDef)]
    return all(isinstance(f.body[-1], (ast.Return, ast.Raise)) for f in fns)

def text_hash(df):
    return hashlib.sha256('\n'.join(df['text_id'].astype(str) + '\n' + df['text'].astype(str)).encode('utf-8')).hexdigest()

def run_checks(df, tokenizers):
    """returns (hard_failures: list[str], warnings: list[str], df_with_counts)"""
    hard, warn = [], []
    df = df.copy()
    # structure
    if len(df) != 400: hard.append(f'rows={len(df)} != 400')
    if df.topic_id.nunique() != 100: hard.append('topics != 100')
    if set(df['style']) != EXPECTED_STYLES: hard.append('style set differs')
    if not df.groupby('topic_id')['style'].nunique().eq(4).all(): hard.append('not 4 styles per topic')
    if not df.text_id.is_unique: hard.append('text_id not unique')
    if df.groupby('topic_id').length_tier.nunique().max() != 1: hard.append('length_tier varies inside a topic')
    if df.groupby('topic_id').operation_id.nunique().max() != 1: hard.append('operation_id varies inside a topic')
    tiers = df.drop_duplicates('topic_id').length_tier.value_counts().to_dict()
    if tiers != {'short': 34, 'medium': 33, 'long': 33}: hard.append(f'tier topic counts {tiers}')
    if not df.pilot_excluded.astype(str).str.lower().isin(['true', '1']).all(): hard.append('pilot_excluded not all True')
    # duplicates (fresh, after all edits)
    norm = df['text'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
    nd = int(norm.duplicated().sum())
    if nd: hard.append(f'{nd} duplicate normalized texts: {df.text_id[norm.duplicated(keep=False)].tolist()}')
    # tokens, both tokenizers, exact band
    for k, tok in tokenizers.items():
        df[f'tokens_{k}'] = df['text'].map(lambda t: len(tok(t, add_special_tokens=True, truncation=False)['input_ids']))
        df[f'in_band_{k}'] = [LENGTH_BANDS[tr][0] <= n <= LENGTH_BANDS[tr][1] for tr, n in zip(df.length_tier, df[f'tokens_{k}'])]
        bad = df.text_id[~df[f'in_band_{k}']].tolist()
        if bad: hard.append(f'{k}: {len(bad)} out of band: {bad[:10]}')
    # code: syntax + completeness
    code = df[df['style'] == 'code_plus_natural_language']
    inv = code.text_id[~code.text.map(is_valid_python)].tolist()
    if inv: hard.append(f'invalid Python: {inv}')
    inc = code.text_id[code.text.map(is_valid_python) & ~code.text.map(function_is_complete)].tolist()
    if inc: hard.append(f'incomplete function (no final return/raise): {inc}')
    # topic fidelity of the replaced items: phrase present, min-max structure present, no number-only twins
    for tid in REPLACE_IDS:
        r = df[df.text_id == f'{tid}_code_plus_natural_language'].iloc[0]
        ph = topic_phrase(df, tid)
        if ph.lower() not in r.text.lower(): hard.append(f'{tid}: topic phrase missing from code text')
        if not ('min(values)' in r.text and 'max(values)' in r.text and '(high-low)' in r.text.replace(' ', '')):
            hard.append(f'{tid}: min-max structure missing (operation_id={r.operation_id})')
    # warnings (not blocking, but must be looked at by the human reviewer)
    def has_domain_word(r): return any(w in r.text.lower() for w in r.topic_domain.lower().split('_') if len(w) > 3)
    nodom = code[~code.apply(has_domain_word, axis=1)]
    if len(nodom): warn.append(f'{len(nodom)} code texts contain no domain word (topic info only via numbers/none): {nodom.text_id.tolist()}')
    tm = code.text.str.replace(r'\d+(\.\d+)?', '#', regex=True).str.lower()
    shared = code[tm.map(tm.value_counts()).gt(1)]
    if len(shared): warn.append(f'{len(shared)} code texts share an identical template with another topic (differ only by numbers)')
    sp = df.groupby('topic_id').agg(**{f'{k}_ratio': (f'tokens_{k}', lambda x: x.max() / x.min()) for k in tokenizers})
    for k in tokenizers:
        n = int((sp[f'{k}_ratio'] > 1.8).sum())
        if n: warn.append(f'{k}: {n} topics with max/min token ratio > 1.8')
    return hard, warn, df


## 4. Apply replacements and run all checks on the edited data

In [ ]:
replacements = build_replacements(manifest_v102)
manifest_v103 = manifest_v102.copy()
manifest_v103['text'] = [replacements.get(i, t) for i, t in zip(manifest_v103.text_id, manifest_v103.text)]

hard, warn, counted = run_checks(manifest_v103, tokenizers)
print('HARD FAILURES:', len(hard))
for h in hard: print('  FAIL', h)
print('WARNINGS (not blocking, human must look):')
for w in warn: print('  WARN', w)
print()
print(counted[counted.text_id.isin(replacements)][['text_id','length_tier'] + [f'tokens_{k}' for k in tokenizers]].to_string(index=False))

## 5. Human review — read each replaced quartet (prose vs. code), then set the flag in step 6

In [ ]:
for tid in REPLACE_IDS:
    s = manifest_v103[manifest_v103.topic_id == tid]
    print('='*100); print(tid, s.topic_domain.iloc[0], '| operation:', s.operation_id.iloc[0], '| tier:', s.length_tier.iloc[0])
    print('[general_prose]  ', s[s['style']=='general_prose'].text.iloc[0][:230], '...')
    print('[code]'); print(s[s['style']=='code_plus_natural_language'].text.iloc[0]); print()

## 6. Freeze (blocked unless hard checks pass AND the flag below is True)

In [ ]:
# ---------- CELL C: freeze v1.0.3 (blocked unless every hard check passes AND a human sets the flag) ----------
HUMAN_REVIEW_OF_REPLACED_ITEMS_AND_SAMPLE_VALUES_DONE = False   # set True only after a person has read the 7 replaced quartets
def freeze_v103(df, tokenizers, source_sha256, out_dir, human_ok):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    hard, warn, df2 = run_checks(df, tokenizers)
    freeze_ok = (not hard) and human_ok
    df2['role'] = 'main_eval_frozen' if freeze_ok else 'candidate_main_eval_not_frozen'   # manifest and report can no longer disagree
    report = {
        'protocol_version': 'N02_Main_Evaluation_Dataset_Design_v1.0.3',
        'rows': int(len(df2)), 'topic_clusters': int(df2.topic_id.nunique()), 'styles_per_topic': 4,
        'tokenizers': TOKENIZER_IDS, 'length_bands': LENGTH_BANDS,
        'hard_check_failures': hard, 'warnings': warn,
        'all_candidates_in_band_for_both_tokenizers': not any('out of band' in h for h in hard),
        'human_review_of_replaced_items_done': bool(human_ok),
        'frozen_for_confirmatory_analysis': bool(freeze_ok),
        'source_manifest_file_sha256': source_sha256,
        'replaced_text_ids': [f'{t}_code_plus_natural_language' for t in REPLACE_IDS],
        'sample_values_source': 'original v1.0 candidate manifest',
        'candidate_text_manifest_sha256': text_hash(df2),
        'role_in_manifest': sorted(df2.role.unique().tolist()),
        'bootstrap_replicates': 10000, 'pilot_prompts_excluded': True,
    }
    path = out_dir / 'N02_Main_Evaluation_FROZEN_Manifest_v1.0.3.csv' if freeze_ok else out_dir / 'N02_Main_Evaluation_CANDIDATE_Manifest_v1.0.3.csv'
    df2.to_csv(path, index=False)
    report['manifest_file'] = path.name
    report['manifest_file_sha256'] = hashlib.sha256(path.read_bytes()).hexdigest()
    (out_dir / 'N02_preflight_report_v1.0.3.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
    return report


In [ ]:
HUMAN_REVIEW_OF_REPLACED_ITEMS_AND_SAMPLE_VALUES_DONE = False   # <- change to True ONLY after step 5 review 
report = freeze_v103(manifest_v103, tokenizers, SOURCE_SHA256, '/content/n02_main_eval_preflight',
                     HUMAN_REVIEW_OF_REPLACED_ITEMS_AND_SAMPLE_VALUES_DONE)
print(json.dumps({k: report[k] for k in ['hard_check_failures','warnings','frozen_for_confirmatory_analysis','role_in_manifest','manifest_file','manifest_file_sha256','candidate_text_manifest_sha256']}, indent=2, ensure_ascii=False))
import shutil; shutil.make_archive('/content/N02_Frozen_Manifest_Audit_v1_0_3', 'zip', '/content/n02_main_eval_preflight')
print('zip: /content/N02_Frozen_Manifest_Audit_v1_0_3.zip')